In [2]:
import os
import psycopg2
import pandas as pd
import warnings

warnings.filterwarnings("ignore")


file_path = "sales_data.csv"  
sales_df = pd.read_csv(file_path)


sales_df = sales_df.loc[:, ~sales_df.columns.str.contains('^Unnamed')]


sales_df.columns = sales_df.columns.str.strip()


sales_df.columns = [col.replace(" ", "_").replace("(", "").replace(")", "").replace("%", "pct") for col in sales_df.columns]


def infer_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    else:
        return "TEXT"


conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="root",
    host="localhost",
    port="5432"
)
cur = conn.cursor()


table_name = "sales_data"
columns = sales_df.dtypes
sql_columns = ",\n  ".join([f'"{col}" {infer_sql_type(dtype)}' for col, dtype in columns.items()])

create_stmt = f"""
CREATE TABLE "{table_name}" (
  {sql_columns}
);
"""

cur.execute(f'DROP TABLE IF EXISTS "{table_name}" CASCADE;')
cur.execute(create_stmt)
conn.commit()
print("Table recreated with schema:")
print(create_stmt)


columns_list = list(sales_df.columns)
placeholders = ', '.join(['%s'] * len(columns_list))
quoted_cols = ', '.join([f'"{col}"' for col in columns_list])

insert_stmt = f'INSERT INTO "{table_name}" ({quoted_cols}) VALUES ({placeholders})'

for _, row in sales_df.iterrows():
    row_values = [None if pd.isna(val) else val for val in row[columns_list]]
    cur.execute(insert_stmt, tuple(row_values))

conn.commit()
print("Data inserted successfully")


df_from_db = pd.read_sql(f'SELECT * FROM "{table_name}"', conn)
cur.close()
conn.close()

print("Data fetched from DB:")
print(df_from_db.head())


Table recreated with schema:

CREATE TABLE "sales_data" (
  "ORDERNUMBER" INT,
  "QUANTITYORDERED" INT,
  "PRICEEACH" FLOAT,
  "ORDERLINENUMBER" INT,
  "SALES" FLOAT,
  "ORDERDATE" TEXT,
  "STATUS" TEXT,
  "QTR_ID" INT,
  "MONTH_ID" INT,
  "YEAR_ID" INT,
  "PRODUCTLINE" TEXT,
  "MSRP" INT,
  "PRODUCTCODE" TEXT,
  "CUSTOMERNAME" TEXT,
  "PHONE" TEXT,
  "ADDRESSLINE1" TEXT,
  "ADDRESSLINE2" TEXT,
  "CITY" TEXT,
  "STATE" TEXT,
  "POSTALCODE" TEXT,
  "COUNTRY" TEXT,
  "TERRITORY" TEXT,
  "CONTACTLASTNAME" TEXT,
  "CONTACTFIRSTNAME" TEXT,
  "DEALSIZE" TEXT,
  "TOTAL_PRICE" FLOAT,
  "MONTH_NAME" TEXT
);

Data inserted successfully
Data fetched from DB:
   ORDERNUMBER  QUANTITYORDERED  PRICEEACH  ORDERLINENUMBER    SALES  \
0        10107               30      95.70                2  2871.00   
1        10121               34      81.35                5  2765.90   
2        10134               41      94.74                2  3884.34   
3        10145               45      83.26              